In [ ]:
from lcdb.db import LCDB

In [ ]:
workflow_mapping = {
  "libsvm": "lcdb.workflow.sklearn.LibSVMWorkflow",
  "randomforest": "lcdb.workflow.sklearn.RandomForestWorkflow",
  "knn": "lcdb.workflow.sklearn.KNNWorkflow",
  "xgboost": "lcdb.workflow.xgboost.XGBoostWorkflow",
  "treesensemble": "lcdb.workflow.sklearn.TreesEnsembleWorkflow",
  "liblinear": "lcdb.workflow.sklearn.LibLinearWorkflow"
}

In [ ]:
workflow = "treesensemble"
workflow_class = workflow_mapping[workflow]

In [ ]:
import pandas as pd

df = pd.read_csv('experiments/surf/snellius/datasets_to_test.csv', header=None)

openmlids = df.iloc[:, 0].tolist()

In [ ]:
import numpy as np
openmlids = np.array(openmlids, dtype=int).tolist()

In [ ]:
lcdb = LCDB()
stats_info = lcdb.statistics(
    # openmlids=[188, 12, 42769],
    openmlids=openmlids,
    workflows=[workflow_class],
    campaigns=['data_probing-57344'],
    validation_seeds=[0],
    test_seeds=[0,1],
    show_progress=True
)

df = stats_info

In [ ]:
import pandas as pd
import numpy as np

# memory from bytes to GB
df["memory_gb"] = df["memory"] / (1024**3)

memory_bins = [0, 1, 2, 4, 8, 16, 24, 32, 40, float("inf")]
memory_labels = ["<1GB", "1GB", "2GB", "4GB", "8GB", "16GB", "24GB", "32GB", "40GB+"]

# memory bins assigned
df["memory_bin"] = pd.cut(df["memory_gb"], bins=memory_bins, labels=memory_labels, right=False)

# group by workflow and memory_bin, listing datasets per bin
grouped_workflow = df.groupby(["workflow", "memory_bin"])["openmlid"].unique().reset_index()

grouped_workflow["openmlid"] = grouped_workflow["openmlid"].apply(lambda x: list(x) if isinstance(x, (np.ndarray, list)) else [])

# remove rows with no openmlids
grouped_workflow = grouped_workflow[grouped_workflow["openmlid"].map(len) > 0]

grouped_workflow


In [ ]:
import pandas as pd

# memory from bytes to GB
df["memory_gb"] = df["memory"] / (1024**3)  

memory_bins = [0, 1, 2, 4, 8, 16, 24, 32, 40, float("inf")]
memory_labels = ["<1GB", "1GB", "2GB", "4GB", "8GB", "16GB", "24GB", "32GB", "40GB+"]

# each row is assigned to memory bin
df["memory_bin"] = pd.cut(df["memory_gb"], bins=memory_bins, labels=memory_labels, right=False)

# grouped by memory bin and collect (workflow, openmlid) pairs as a list
grouped = df.groupby("memory_bin").agg(
    workflow_dataset_combinations=("workflow", lambda x: list(set(zip(x, df.loc[x.index, "openmlid"]))))
).reset_index()

# remove bins with no tuples
grouped = grouped[grouped["workflow_dataset_combinations"].map(len) > 0]

grouped
